# 16A Appendix: Backprop by Hand, and More scikit-learn Examples


[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tools4ds/DS701-Materials-FA26/blob/main/jupyter_notebooks/16A-Appendix-NN-Walkthroughs.ipynb)

In [ ]:
# Setup for Colab and for a local clone of the Materials repo: make the course's
# helper modules importable, downloading any that are not already available.
import importlib.util, os, sys, urllib.request

_BASE = "https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main"
for _path in ["scripts/slideUtilities.py"]:
    _name = os.path.splitext(os.path.basename(_path))[0]
    if importlib.util.find_spec(_name) is not None:
        continue
    for _root in (".", ".."):                      # a clone: notebook in jupyter_notebooks/
        if os.path.exists(os.path.join(_root, _path)):
            sys.path.insert(0, os.path.abspath(os.path.dirname(os.path.join(_root, _path))))
            break
    else:                                          # Colab: fetch the file
        os.makedirs("modules", exist_ok=True)
        urllib.request.urlretrieve(f"{_BASE}/{_path}", os.path.join("modules", os.path.basename(_path)))
        if os.path.abspath("modules") not in sys.path:
            sys.path.insert(0, os.path.abspath("modules"))


In [ ]:
#| code-fold: true
#| code-summary: "Imports and setup"
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.exceptions import ConvergenceWarning
from slideUtilities import OKABE_ITO

warnings.filterwarnings('ignore')
%matplotlib inline

This appendix holds the worked material behind [Lecture 16](16-Neural-Networks.qmd). The first half fills in every gradient on the lecture's tiny graph by hand, then does the same for a ReLU neuron and for the smallest network with a loss on top, takes one gradient step, and shows a dead ReLU. The second half is the scikit-learn material the lecture only mentions: a regression example on the California-housing data, grid search, solver choice, and what scikit-learn's MLP cannot do. None of it is needed for the quiz, which asks you to recognize what backpropagation computes and to read training curves, not to run backprop by hand.

**Note**

## Where the `Value` class comes from
The `Value` class below is a condensed version of the one [Module M12](M12-NN-Backprop.qmd) builds step by step: it starts from a bare wrapper around a number, adds `+` and `*`, records child nodes and operations, draws the graph, and only then computes gradients. M12 also adds an automatic `backward()`, handles nodes used more than once (gradients *accumulate*), assembles neurons into layers and an MLP, writes a training loop, and does the same in PyTorch. Here the finished class is simply given, because the point of this appendix is the gradients, not the class.


# Backprop by Hand

## The `Value` Class

The class has four parts: the stored number (`data`), a gradient slot (`grad`), a record of how the node was made (`_prev`, `_op`), and the operations `+`, `*`, `relu()` and `loss()`. Each operation returns a **new** `Value` that remembers its operands, which is how the graph builds itself. It is a miniature of PyTorch's `Tensor`.

In [ ]:
#| code-fold: false
class Value:

    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0               # dL/d(this node), filled in by backprop
        self._prev = set(_children)   # the operand nodes
        self._op = _op                # the operation that created this node
        self.label = label            # a name, for drawing

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):         # self + other
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), '+')

    def __mul__(self, other):         # self * other
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data * other.data, (self, other), '*')

    def relu(self):
        return Value(max(0.0, self.data), (self,), 'ReLU')

    def loss(self, y):                # squared-error loss 1/2 (self - y)^2
        return Value(0.5 * (self.data - y) ** 2, (self,), '½(ŷ−y)²')

## Drawing the Compute Graph

Two helpers walk the graph to collect nodes and edges (`trace`) and draw it (`draw_dot`).

In [ ]:
#| code-fold: true
#| code-summary: "Show the code for `trace()` and `draw_dot()`"
from graphviz import Digraph

def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root):
    dot = Digraph(format='svg',
                  graph_attr={'rankdir': 'LR', 'ranksep': '0.3', 'nodesep': '0.15',
                              'size': '11,4'},
                  node_attr={'fontsize': '20'})
    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        # one box per Value, with its name, data and grad stacked
        # (+ 0.0 turns a negative zero into a plain 0)
        dot.node(name=uid, shape='record',
                 label="%s | data %.4g | grad %.4g" % (n.label, n.data + 0.0, n.grad + 0.0))
        if n._op:
            dot.node(name=uid + n._op, label=n._op)
            dot.edge(uid + n._op, uid)
    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)
    return dot

The lecture's three-stage graph, ending in a node called `L`:

In [ ]:
#| code-fold: false
a = Value(4.0, label='a')
b = Value(-3.0, label='b')
c = Value(8.0, label='c')

d = a*b; d.label = 'd'
e = d + c; e.label = 'e'
f = Value(2.0, label='f')

L = e*f; L.label = 'L'
draw_dot(L)

Computing the `data` values left to right is the **forward pass**. The gradient slots are all still zero.

## Manual Gradient Calculation

For the output node, trivially $dL/dL = 1$:

In [ ]:
#| code-fold: false
L.grad = 1.0

One step back, $L = e \times f$, so

$$
\frac{\partial L}{\partial e} = f, \qquad \frac{\partial L}{\partial f} = e.
$$

For a **product**, the partial with respect to one operand is the *other operand*. We needed `e.data` and `f.data` to say that, and those come from the forward pass, which is why the forward pass must run first.

In [ ]:
#| code-fold: false
e.grad = f.data
f.grad = e.data
draw_dot(L)

Sanity check. `f.grad` says $L$ should change by `e.data` $= -4$ for a unit change in $f$. `wiggle` rebuilds the graph with one input raised by $h$ and reports the slope:

In [ ]:
#| code-fold: false
def wiggle(build, params, var, h=1.0):
    L0 = build(**params).data
    L1 = build(**{**params, var: params[var] + h}).data
    print(f"L: {L0:g} -> {L1:g} when {var} goes up by {h:g};  slope = {(L1 - L0) / h:g}")

def toy_graph(a, b, c, f):
    return (Value(a) * Value(b) + Value(c)) * Value(f)

toy = dict(a=4.0, b=-3.0, c=8.0, f=2.0)
wiggle(toy_graph, toy, 'f')

## Propagating Back

Now $\partial L/\partial c$. In the graph, $c$ influences $e$ and $e$ influences $L$, and $e = d + c$ has the **local** derivative $\partial e/\partial c = 1$. The chain rule puts the two together:

$$
\frac{\partial L}{\partial c} = \frac{\partial L}{\partial e} \cdot \frac{\partial e}{\partial c},
$$

each factor evaluated at the forward-pass values. Since the local derivative of a sum is $1$ with respect to either operand, the **addition node just routes the parent gradient to both children**:

In [ ]:
#| code-fold: false
d.grad = e.grad
c.grad = e.grad
draw_dot(L)

## Propagating Back Again

One more step, from $d$ to $a$ and $b$. Since $d = a \cdot b$, the local derivatives are $\partial d/\partial b = a$ and $\partial d/\partial a = b$, so

$$
\frac{\partial L}{\partial b} = \frac{\partial L}{\partial d} \cdot a, \qquad
\frac{\partial L}{\partial a} = \frac{\partial L}{\partial d} \cdot b.
$$

Fully expanded, $\partial L/\partial b$ is a chain of local derivatives all the way from $L$:

$$
\frac{\partial L}{\partial b}
  = \frac{\partial L}{\partial e} \cdot \frac{\partial e}{\partial d} \cdot \frac{\partial d}{\partial b}
  = 2 \cdot 1 \cdot 4 = 8.
$$

In [ ]:
#| code-fold: false
b.grad = a.data * d.grad
a.grad = b.data * d.grad
draw_dot(L)

Check: `b.grad` $= 8$, so raising $b$ by $1$ should raise $L$ by $8$:

In [ ]:
#| code-fold: false
wiggle(toy_graph, toy, 'b')

## Backprop on a Neuron

The same recipe applies to a real neuron: two inputs, two weights, a bias, and a ReLU.

In [ ]:
#| code-fold: false
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')
b = Value(7.0, label='b')

x1w1 = x1*w1; x1w1.label = 'x1*w1'
x2w2 = x2*w2; x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1w1 + x2w2'
n = x1w1x2w2 + b; n.label = 'n'
o = n.relu(); o.label = 'o'
draw_dot(o)

The only new operation is the ReLU. It is not differentiable at $0$, but in practice we take the derivative to be $0$ when the input is $\le 0$ and $1$ when it is $> 0$.

<!-- Credit: _Understanding Deep Learning_, Figure 7.6 -->
![](https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main/figs/NN-figs/Train2ReLUDeriv.svg)

In [ ]:
#| code-fold: false
o.grad = 1.0
n.grad = o.grad if n.data > 0 else 0.0   # ReLU: pass through if the input was positive
x1w1x2w2.grad = n.grad                    # '+' routes the gradient
b.grad = n.grad
x1w1.grad = x1w1x2w2.grad
x2w2.grad = x1w1x2w2.grad
w1.grad = x1.data * x1w1.grad             # '*' multiplies by the other operand
w2.grad = x2.data * x2w2.grad
draw_dot(o)

Note `w2.grad` $= 0$ because $x_2 = 0$: this weight has *no* influence on the output for this input. That is the `*` rule at work, a zero *input* to a product.

## A Network with a Loss

Real training puts a **loss** on top. Here is the smallest network with everything: one input, one hidden ReLU unit, one output, and the squared-error loss.

$$
z_1 = w_1 x + b_1,\qquad h = \mathrm{ReLU}(z_1),\qquad \hat y = w_2 h + b_2,\qquad L = \tfrac12(\hat y - y)^2
$$

with $w_1 = 1,\ b_1 = 0.5,\ w_2 = -2,\ b_2 = 1$ and one training example $x = 1,\ y = 0$. **Forward pass** by hand: $z_1 = 1.5$, $h = 1.5$, $\hat y = -2$, $L = 2$.

In [ ]:
#| code-fold: false
from types import SimpleNamespace

def two_layer(w1, b1, w2, b2, x, y):
    g = SimpleNamespace(y=y)
    g.x, g.w1, g.b1 = Value(x, label='x'), Value(w1, label='w1'), Value(b1, label='b1')
    g.w2, g.b2 = Value(w2, label='w2'), Value(b2, label='b2')
    g.w1x = g.w1 * g.x;     g.w1x.label = 'w1*x'
    g.z1 = g.w1x + g.b1;    g.z1.label = 'z1'
    g.h = g.z1.relu();      g.h.label = 'h'
    g.w2h = g.w2 * g.h;     g.w2h.label = 'w2*h'
    g.yhat = g.w2h + g.b2;  g.yhat.label = 'yhat'
    g.L = g.yhat.loss(y);   g.L.label = 'L'
    return g

params = dict(w1=1.0, b1=0.5, w2=-2.0, b2=1.0)
net = two_layer(**params, x=1.0, y=0.0)
draw_dot(net.L)

## Backward Through the Loss

The recipe, node by node from the top. The loss node's local derivative is $\partial L/\partial\hat y = \hat y - y$:

In [ ]:
#| code-fold: false
def backward_by_hand(g):
    g.L.grad = 1.0
    g.yhat.grad = (g.yhat.data - g.y) * g.L.grad      # loss node: send yhat - y
    g.w2h.grad = g.yhat.grad                          # '+' routes
    g.b2.grad = g.yhat.grad
    g.w2.grad = g.h.data * g.w2h.grad                 # '*' multiplies by the other operand
    g.h.grad = g.w2.data * g.w2h.grad
    g.z1.grad = g.h.grad if g.z1.data > 0 else 0.0    # ReLU gates
    g.w1x.grad = g.z1.grad                            # '+' routes
    g.b1.grad = g.z1.grad
    g.w1.grad = g.x.data * g.w1x.grad                 # '*' again
    g.x.grad = g.w1.data * g.w1x.grad                 # (inputs get one too; unused)

backward_by_hand(net)
draw_dot(net.L)

Reading the numbers off the graph, top to bottom:

* loss node: $\partial L/\partial\hat y = \hat y - y = -2$
* `+` routes it: $\partial L/\partial b_2 = -2$
* `*` swaps: $\partial L/\partial w_2 = h \cdot (-2) = -3$, and $\partial L/\partial h = w_2 \cdot (-2) = 4$
* ReLU: $z_1 = 1.5 > 0$, so it passes $4$ through: $\partial L/\partial z_1 = 4$
* `+` routes: $\partial L/\partial b_1 = 4$; `*` swaps: $\partial L/\partial w_1 = x \cdot 4 = 4$

## The Chain from $L$ to $w_1$

Written as one product of local derivatives, from the loss back to the first weight:

$$
\frac{\partial L}{\partial w_1}
= \underbrace{\frac{\partial L}{\partial \hat y}}_{\hat y - y}
  \cdot \underbrace{\frac{\partial \hat y}{\partial h}}_{w_2}
  \cdot \underbrace{\frac{\partial h}{\partial z_1}}_{\text{ReLU}'(z_1)}
  \cdot \underbrace{\frac{\partial z_1}{\partial w_1}}_{x}
= (-2)(-2)(1)(1) = 4.
$$

So raising $w_1$ by $0.01$ should raise the loss by about $0.04$:

In [ ]:
#| code-fold: false
def net_loss(**p):
    return two_layer(**p).L

wiggle(net_loss, dict(**params, x=1.0, y=0.0), 'w1', h=0.01)

The finite difference, $4.02$, agrees with backprop's $4$; the extra $0.02$ is the curvature a step of $h = 0.01$ picks up. This finite-difference check is what the numerical "wiggle" is still good for: testing a backprop implementation.

## One Gradient Step

One step of gradient descent with $\eta = 0.05$ on each parameter leaf, $w \leftarrow w - \eta\,\partial L/\partial w$, then a fresh forward pass:

In [ ]:
#| code-fold: false
eta = 0.05
new_params = {k: v - eta * getattr(net, k).grad for k, v in params.items()}
new_net = two_layer(**new_params, x=1.0, y=0.0)

print({k: round(v, 3) for k, v in new_params.items()})
print(f"yhat: {net.yhat.data:g} -> {new_net.yhat.data:g}    "
      f"loss: {net.L.data:g} -> {new_net.L.data:.4g}")

Every parameter moved *against* its gradient: $w_1 = 1 - 0.05(4) = 0.8$, $b_1 = 0.5 - 0.05(4) = 0.3$, $w_2 = -2 - 0.05(-3) = -1.85$, $b_2 = 1 - 0.05(-2) = 1.1$. The prediction moved from $-2$ toward the target $0$, and the loss fell from $2$ to $0.437$. That is one iteration of training.

## A Dead ReLU

Same parameters, but flip the sign of the input: $x = -1$. Now $z_1 = -0.5 < 0$, so $h = 0$, $\hat y = b_2 = 1$ and $L = \tfrac12$.

In [ ]:
#| code-fold: false
dead = two_layer(**params, x=-1.0, y=0.0)
backward_by_hand(dead)
draw_dot(dead.L)

Two different zeros, from two different rules:

* $\partial L/\partial w_2 = h \cdot 1 = 0$: the `*` rule, because $w_2$'s *input* $h$ is $0$, the same mechanism as `w2.grad` $= 0$ in the neuron example.
* $\partial L/\partial z_1 = 0$, so $w_1$ **and** $b_1$ get $0$: the **ReLU gate** is closed, and it zeroes *everything upstream*, even though $\partial L/\partial h = -2$ is not zero.
* Only $b_2$ learns from this example.

If *every* training example put the unit in this state, $w_1$ and $b_1$ would never change and the unit would output $0$ forever: a **dead** (or "dying") **ReLU**. A too-large step can push a unit into this state, which is one reason the learning rate and initialisation matter, and why variants such as leaky ReLU exist.

# More scikit-learn Examples

## Regression: California Housing

The lecture's classifier has a regression twin, `MLPRegressor`. Here it predicts the median house value (in \$100,000s) of a California district from eight features, with early stopping switched on.

In [ ]:
#| code-fold: false
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor

housing = fetch_california_housing()
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    housing.data, housing.target, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_h_scaled = scaler.fit_transform(X_train_h)
X_test_h_scaled = scaler.transform(X_test_h)

mlp_reg = MLPRegressor(hidden_layer_sizes=(100, 50), activation='relu', solver='adam',
                       alpha=0.001, batch_size=32, max_iter=100,
                       early_stopping=True, validation_fraction=0.1, n_iter_no_change=10,
                       random_state=42)
mlp_reg.fit(X_train_h_scaled, y_train_h)
print(f"features: {housing.feature_names}")
print(f"training stopped at epoch {mlp_reg.n_iter_} of max_iter={mlp_reg.max_iter}")
print(f"best validation R²: {mlp_reg.best_validation_score_:.4f}")

With `early_stopping=True`, scikit-learn holds out 10 % of the training rows, scores them ($R^2$) after every epoch, stops once that score has not improved for 10 epochs, and keeps the best weights.

## Training and Validation Curves

In [ ]:
#| code-fold: true
#| fig-align: center
epochs = np.arange(1, mlp_reg.n_iter_ + 1)
best_epoch_h = int(np.argmax(mlp_reg.validation_scores_)) + 1

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, mlp_reg.loss_curve_, color=OKABE_ITO[5], lw=2)
axes[0].set_xlabel('epoch')
axes[0].set_ylabel('training loss (½ MSE + L2 penalty)')
axes[0].set_title('Housing MLP: training loss')
axes[0].grid(alpha=0.3)

axes[1].plot(epochs, mlp_reg.validation_scores_, color=OKABE_ITO[0], lw=2)
axes[1].axvline(best_epoch_h, color='gray', ls='--',
                label=f'best validation epoch ({best_epoch_h})')
axes[1].set_xlabel('epoch')
axes[1].set_ylabel('validation $R^2$ (10% held out)')
axes[1].set_title('Housing MLP: validation score')
axes[1].legend(loc='lower right')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

The training loss is noisy because each step uses a mini-batch of 32 rows (`adam` recovers from the bumps). The validation score is what early stopping watches: it peaked at the dashed line, and after 10 more epochs without improvement training stopped.

## Evaluating the Regressor

In [ ]:
#| code-fold: false
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

y_pred_h = mlp_reg.predict(X_test_h_scaled)
rmse = np.sqrt(mean_squared_error(y_test_h, y_pred_h))
mae = mean_absolute_error(y_test_h, y_pred_h)
r2 = r2_score(y_test_h, y_pred_h)
print(f"RMSE: {rmse:.4f}   MAE: {mae:.4f}   R²: {r2:.4f}")

An $R^2$ of about $0.8$ means the model explains 80 % of the variance in house prices, relative to always predicting the mean; RMSE and MAE are in the target's units, so they say how far off a typical prediction is.

In [ ]:
#| code-fold: true
#| fig-align: center
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))

axes[0].scatter(y_test_h, y_pred_h, alpha=0.4, s=8, color=OKABE_ITO[5])
axes[0].plot([y_test_h.min(), y_test_h.max()],
             [y_test_h.min(), y_test_h.max()], color=OKABE_ITO[3], ls='--', lw=2)
axes[0].set_xlabel('Actual value ($100k)')
axes[0].set_ylabel('Predicted value ($100k)')
axes[0].set_title('Predicted vs actual')
axes[0].grid(alpha=0.3)

residuals = y_test_h - y_pred_h
axes[1].scatter(y_pred_h, residuals, alpha=0.4, s=8, color=OKABE_ITO[5])
axes[1].axhline(y=0, color=OKABE_ITO[3], linestyle='--', lw=2)
axes[1].set_xlabel('Predicted value ($100k)')
axes[1].set_ylabel('Residual ($100k)')
axes[1].set_title('Residuals')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

The vertical wall at 5 and the diagonal streak in the residuals are not the network's doing: the source data caps the target at \$500k, so every district worth more is recorded as 5.0 (about 5 % of them).

## Why Scaling Matters: California Housing

The same `(64,)` network on housing, fitted on standardized and on raw features with a fixed 100-epoch budget:

In [ ]:
#| code-fold: false
fits = {}
for name, (A_tr, A_te) in {'standardized': (X_train_h_scaled, X_test_h_scaled),
                           'raw': (X_train_h, X_test_h)}.items():
    m = MLPRegressor(hidden_layer_sizes=(64,), max_iter=100, random_state=42)
    m.fit(A_tr, y_train_h)
    fits[name] = m
    print(f"{name:12s}: epochs {m.n_iter_:3d}   first-epoch loss {m.loss_curve_[0]:9.2f}"
          f"   test R² {m.score(A_te, y_test_h):.3f}")

In [ ]:
#| code-fold: true
#| fig-align: center
fig, ax = plt.subplots(figsize=(10, 4))
for (name, m), c in zip(fits.items(), [OKABE_ITO[5], OKABE_ITO[3]]):
    ax.plot(np.arange(1, m.n_iter_ + 1), m.loss_curve_, color=c, lw=2, label=name)
ax.set_yscale('log')
ax.set_xlabel('epoch')
ax.set_ylabel('training loss (log scale)')
ax.set_title('Housing, (64,) MLP: standardized vs raw features')
ax.legend()
ax.grid(alpha=0.3, which='both')
plt.show()

Raw `Population` runs into the tens of thousands while `MedInc` is near 4, so a few columns dominate every weighted sum and every gradient. The raw fit starts with an enormous loss, stalls, and stops "because it stopped improving" at a much worse $R^2$.

## Grid Search

[`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) tries every combination in a parameter grid with cross-validation. On the lecture's 8×8 digits, three architectures times two values of `alpha`, three folds each:

In [ ]:
#| code-fold: false
from sklearn.datasets import load_digits
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV

SEED = 701
digits = load_digits()
X_train, X_test, y_train, y_test = train_test_split(
    digits.data, digits.target, test_size=0.3, stratify=digits.target, random_state=SEED)
scaler_d = StandardScaler().fit(X_train)
X_train_s, X_test_s = scaler_d.transform(X_train), scaler_d.transform(X_test)

mlp_grid = MLPClassifier(max_iter=60, early_stopping=True, validation_fraction=0.1,
                         n_iter_no_change=5, random_state=SEED)
param_grid = {'hidden_layer_sizes': [(50,), (100,), (50, 50)],
              'alpha': [0.0001, 0.001]}

grid_search = GridSearchCV(mlp_grid, param_grid, cv=3, n_jobs=2)
grid_search.fit(X_train_s, y_train)
print(f"best parameters: {grid_search.best_params_}")
print(f"best CV accuracy: {grid_search.best_score_:.4f}")

## Grid Search Results

In [ ]:
#| code-fold: true
results_df = pd.DataFrame(grid_search.cv_results_)
ranked = (results_df
          .sort_values('rank_test_score')
          .assign(hidden_layer_sizes=lambda d: d['param_hidden_layer_sizes'].astype(str),
                  alpha=lambda d: d['param_alpha'])
          [['rank_test_score', 'hidden_layer_sizes', 'alpha',
            'mean_test_score', 'std_test_score']]
          .rename(columns={'rank_test_score': 'rank',
                           'mean_test_score': 'mean CV accuracy',
                           'std_test_score': 'std across folds'}))
best = ranked.iloc[0]
within = (ranked['mean CV accuracy'] >= best['mean CV accuracy'] - best['std across folds']).sum()
print(ranked.to_string(index=False, float_format='{:.4f}'.format))
print(f"\nconfigurations within one std of the best: {within} of {len(ranked)}")

Read the spread, not just the winner: $3 \times 2 = 6$ configurations, each fitted three times, is 18 training runs before the final refit, and when several configurations sit within one standard deviation of the best, the grid cannot tell them apart. Pick the simplest. Tune on the folds, never on the test set; the test set is spent once, at the end.

## Solver Selection

| solver | best for | notes |
|--------|----------|-------|
| `'adam'` | most cases | good default, adaptive per-parameter step size |
| `'sgd'` | large datasets | classic mini-batch SGD; needs the learning rate set well |
| `'lbfgs'` | small datasets | full-batch quasi-Newton; fast on small data, memory-hungry; no `loss_curve_` |

In [ ]:
#| code-fold: true
for solver in ['adam', 'sgd', 'lbfgs']:
    m = MLPClassifier(hidden_layer_sizes=(50,), solver=solver, max_iter=50, random_state=SEED)
    m.fit(X_train_s, y_train)
    print(f"{solver:6s}: test accuracy {m.score(X_test_s, y_test):.4f}   final training loss {m.loss_:.4f}")

`sgd` trails because its default `learning_rate_init=0.001` is too small for a 50-epoch budget: its training loss is still high, the lecture's "too low" picture. For `lbfgs`, `max_iter` counts full-batch iterations, not epochs, so this is not a like-for-like budget.

## What scikit-learn's MLP Cannot Do

scikit-learn's MLP is a fully connected network with a fixed recipe. It initializes weights with Glorot (Xavier) scaling and offers L2 and early stopping, but nothing else:

* no **dropout** or **batch normalization**
* no learning-rate **warm-up** or custom schedules (beyond `sgd`'s three; see [Module M5](M5-NN-Scikit-Learn.qmd#learning-rate-schedules))
* no **GPU**, no custom losses or layers
* no **convolutional** or **recurrent** layers for images and sequences

Use it for small-to-medium tabular data, standard feed-forward architectures, CPU training, and anything that should live inside a scikit-learn `Pipeline` with cross-validation and grid search. For large data, GPUs, CNNs, RNNs and transformers, move to PyTorch: see [Module M4: CNNs](M4-CNNs.qmd), [Module M3: RNNs](M3-RNN.qmd), the PyTorch comparison in [Module M5](M5-NN-Scikit-Learn.qmd#comparison-scikit-learn-vs-pytorch), and DS542, Deep Learning for Data Science.